# Oscillating Droplet (2D)

A circular droplet of radius `R = 1`, given the straining velocity field
`v = (A x, -A y)` and held together by a radial potential of strength `B`. The
strain stretches it into an ellipse, the potential pulls it back, and it
oscillates -- and because the flow is (nearly) incompressible the ellipse it
passes through has the same area as the circle it started as. Both the
**period** and the **amplitude** have closed forms, which is what makes this a
test rather than a demo:

- period `T = 4.827 / A` (`DROPLET_PERIOD`, exported by the case),
- extreme semi-axes `1.931843 R` and `R / 1.931843` (`DROPLET_STRETCH`,
  likewise, via `analyticEnvelope`).

The run below is exactly one period, so the droplet should come back to a
circle at the last frame. The last cell overlays all three analytic outlines --
the initial circle and the two extreme ellipses -- on the final particles;
`tLimit = T/4` instead puts the droplet on the wide ellipse, which is the
easier comparison to read.

The potential is configured, not gravity: `configureScheme` sets
`gravityType = 'PotentialField'` with magnitude `B` and origin `[0, 0]` before
deferring to the shared setup, so `--gravityMagnitude` is `B` here rather than
`9.81` pointing down.

![](outputs/04-oscillatingDroplet.gif)


## Every knob, and what it does

The parameters cell below is the whole command line of `04-oscillating-droplet.py` written out:
`CaseSpec` fields first, then `oscillatingDropletCase.params` -- the case's own physics knobs,
each of which is also a `--flag`. Anything not named there keeps the value in
`oscillatingDropletCase.defaults`/`.params`.

**Discretisation, time stepping and output** (`CaseSpec` fields, shared by every case)

| field | this notebook | what it does |
|---|---|---|
| `nx` | `192` | particles across the domain; the spacing is `dx = L / nx` |
| `dim` | `2` | this case is 2D |
| `L` | `6.0` | side of the (periodic) box |
| `n_h` | `4.0` | particles per support radius, i.e. how smooth the kernel is |
| `kernel` | `Wendland4` | SPH kernel |
| `integrationScheme` | `rungeKutta2` | time integrator |
| `scheme` | `deltaSPH` | the solver itself |
| `tLimit` | `4.827 (one period)` | simulated end time; the loop runs `tLimit / dt` steps |
| `dt` | *set by the case* | left `None`: `initialConditions` picks it together with the sound speed |
| `adaptiveDt`, `cflFactor`, `minDt` | `True`, `0.3`, `1e-8` | CFL limiter around that `dt` |
| `plot`, `show`, `plotInterval` | `True`, `True`, `10` | render a frame every `plotInterval` steps |
| `store`, `storeMode`, `storeInterval` | `False`, `'states'`, `500` | HDF5 export; off here |

**The case's own parameters** (`--flag` on the script, `params=dict(...)` here)

| parameter | this notebook | what it does |
|---|---|---|
| `R` | `1.0` | initial droplet radius |
| `A` | `1.0` | strain rate of the initial velocity field `(A x, -A y)`; sets the period, `T = 4.827 / A` |
| `B` | `1.0` | strength of the central potential holding the droplet together |
| `rho0` | `1.0` | rest density |
| `targetDt` | `0.00025` | the timestep the run *asks* for; the sound speed is then chosen to make it the acoustic CFL limit |
| `inviscid`, `nu` | `True`, `0.0` | physical viscosity: `inviscid=True` leaves the scheme's own dissipation as the only one |
| `freeSurface` | `True` | surface detection, on for a case with a free surface |
| `band` | `0` | particle layers of boundary padding around the domain |
| `markerSize` | `8` | plot only: particle marker size |


**Three things this family does differently from the compressible notebooks**
(they will bite if `../compressible/08-Hydrostatic.ipynb` is copied unread):

1. The IC cell has a **fourth call**, `oscillatingDropletCase.initialConditions(ctx, system)`.
   That is where `setupWeaklyCompressibleTimestep` picks the sound speed and
   `config.dt` *together* from `targetDt` -- weakly compressible SPH is free to
   choose its own stiffness, so the timestep is fixed first and `c0` follows
   from the acoustic CFL. Skip it and `config.dt` stays `None`, and the droplet never gets its straining velocity field, which is stamped on in the same call.
2. **The loop is `range(nSteps)`.** No case in this family has a `timestep`
   hook, so `dt` is fixed for the whole run after step 1 and `while t < tLimit`
   would be the wrong shape.
3. Plotting calls `buildFieldPlotter`/`refreshFieldPlotter` on `VELOCITY_DENSITY_FIELDS`
   directly rather than `oscillatingDropletCase.setupPlot`/`updatePlot`, which go through
   `openWindow`/`pumpEvents` and do not live-update inside a Jupyter cell in
   this environment -- `08-Hydrostatic.ipynb` explains that in full.

Precision note: switching between single and double precision is controlled in
the import cell below. Because precision is set when core modules/kernels are
initialized, any precision change requires a kernel restart.

In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.oscillatingDroplet import oscillatingDropletCase, DROPLET_PERIOD, DROPLET_STRETCH, analyticEnvelope
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame
from warpSPH.cases.weaklyCompressible import VELOCITY_DENSITY_FIELDS

import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `04-oscillating-droplet.py`, made explicit and editable here -- the table in the
# intro cell says what each one does. `oscillatingDropletCase.defaults`/`.params` are
# the same values the CLI script starts from.
spec = CaseSpec(caseName=oscillatingDropletCase.name, scheme=oscillatingDropletCase.scheme,
                params=dict(oscillatingDropletCase.params)) \
    .merged(**oscillatingDropletCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=192,
    dim=2,
    # The droplet is 2R across in a 6-wide periodic box: it stretches to
    # ~3.9R at the extremes and must not see its own image.
    L=6.0,

    # --- time stepping ---------------------------------------------------
    # Exactly one oscillation period, DROPLET_PERIOD / A. Use
    # `DROPLET_PERIOD / 4` to stop on the wide ellipse instead.
    tLimit=DROPLET_PERIOD,

    # --- output --------------------------------------------------------------
    caseName='04-oscillatingDroplet',
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- the droplet's own knobs ----------------------------------------------
    params=dict(
        # the droplet, its strain and the potential holding it together
        R=1.0, A=1.0, B=1.0,
        # the fluid
        freeSurface=True, rho0=1.0, targetDt=0.00025, inviscid=True, nu=0.0,
        markerSize=8,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`oscillatingDropletCase.buildSystem`), not re-derived here.
#
# `initialConditions` is the call the compressible notebooks do not have:
# the straining velocity field is stamped on there, and
# it is where the sound speed and `config.dt` are chosen together from
# `targetDt`, so skipping it leaves `config.dt` unset.
ctx = buildContext(oscillatingDropletCase, spec)
oscillatingDropletCase.configureScheme(ctx)
system = oscillatingDropletCase.buildSystem(ctx)
oscillatingDropletCase.initialConditions(ctx, system)
runningState = system.initializeNewState()

print(f'dt = {float(ctx.config.dt):.3e}, '
      f'c0 = {ctx.schemeConfig.fluid.fixedSoundSpeed:.3f}, '
      f'{len(runningState.state.positions)} particles')

In [ ]:
# What was actually built: the sampled regions, fluid and boundary, against the
# domain (black) the run is periodic in. This is the cell to look at when a
# geometry parameter above did something other than what it sounded like.
figure, axis = plt.subplots(1, 1, figsize=(5, 5), squeeze=False)
plotRegions(ctx.scratch['regions'], axis[0, 0], plotFluid=True, plotParticles=True)
domain = ctx.config.domain
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(domain.min[0].item(), domain.max[0].item())
axis[0, 0].set_ylim(domain.min[1].item(), domain.max[1].item())
axis[0, 0].set_title(f'{len(ctx.scratch["regions"])} regions, '
                     f'{len(runningState.state.positions)} particles')
figure.tight_layout()

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(VELOCITY_DENSITY_FIELDS), not oscillatingDropletCase.setupPlot -- see the intro cell
# for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, VELOCITY_DENSITY_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = oscillatingDropletCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=oscillatingDropletCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
extent = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    # Injected at the hook point: the droplet's half-width and half-height,
    # which is the quantity the analytic envelope is a statement about and
    # which no diagnostic records.
    extent.append(runningState.state.positions.abs().max(dim=0).values.tolist())
    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = oscillatingDropletCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, VELOCITY_DENSITY_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=oscillatingDropletCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## Against the analytic solution

Two checks, in order of how much they tell you: the shape at the end of the run
against the analytic envelope, and the extent over time against the extremes it
is supposed to reach.

In [ ]:
# The analytic envelope over the final particles: the circle it started as
# (red) and the two extreme ellipses (cyan), both area-conserving. The
# constants come from the case (`analyticEnvelope`), not from a literal here --
# the original notebook drew the wide ellipse 2R x R, which is not the same
# area as the circle.
import matplotlib.patches as patches

R = spec.param('R')
long_, short = analyticEnvelope(R)

figure, axis = plt.subplots(1, 1, figsize=(5.5, 5.5), squeeze=False)
positions = runningState.state.positions.detach().cpu().numpy()
scatter = axis[0, 0].scatter(positions[:, 0], positions[:, 1], s=1, alpha=0.5,
                             c=runningState.state.densities.detach().cpu().numpy())
figure.colorbar(scatter, ax=axis[0, 0], label='density')
for width, height, color in ((2 * R, 2 * R, 'red'),
                             (2 * long_, 2 * short, 'cyan'),
                             (2 * short, 2 * long_, 'cyan')):
    axis[0, 0].add_patch(patches.Ellipse((0, 0), width, height, fill=False,
                                         edgecolor=color, ls='--', lw=1))
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(-2.05 * R, 2.05 * R)
axis[0, 0].set_ylim(-2.05 * R, 2.05 * R)
axis[0, 0].set_title(f't = {float(runningState.t):.3f} of T = {DROPLET_PERIOD / spec.param("A"):.3f}')
figure.tight_layout()

In [ ]:
# The oscillation itself: the droplet's half-extent in x and y against the
# analytic extremes. One period of `tLimit` should bring both back to R.
figure, axis = plt.subplots(1, 1, figsize=(7, 3.5))
t = [row['t'] for row in trajectory]
extentArray = np.array(extent)
axis.plot(t, extentArray[:, 0], label='half-width')
axis.plot(t, extentArray[:, 1], label='half-height')
for value, label in ((R, '$R$'), (long_, r'$1.93\,R$'), (short, r'$R/1.93$')):
    axis.axhline(value, color='black', ls=':', lw=0.8)
    axis.annotate(label, (t[0], value), fontsize=8, va='bottom')
axis.set_xlabel('t'); axis.set_ylabel('extent'); axis.legend()
figure.tight_layout()

## The usual weakly compressible check

In [ ]:
# The two numbers worth reading off any weakly compressible run: the density
# has to stay within about a percent of `rho0` (that is the whole premise of
# the scheme), and the kinetic energy says whether the flow is doing what it
# was set up to do. Both come from `weaklyCompressibleDiagnostics`, recorded
# every step in the loop above.
figure, axis = plt.subplots(1, 2, figsize=(11, 3.5))
t = [row['t'] for row in trajectory]
axis[0].plot(t, [row['maxDensity'] for row in trajectory], label='max')
axis[0].plot(t, [row['minDensity'] for row in trajectory], label='min')
axis[0].axhspan(0.99, 1.01, color='green', alpha=0.1, label=r'$\pm 1\%$')
axis[0].set_xlabel('t'); axis[0].set_ylabel(r'$\rho$'); axis[0].legend()
axis[1].plot(t, [row['kineticEnergy'] for row in trajectory])
axis[1].set_xlabel('t'); axis[1].set_ylabel('kinetic energy')
figure.tight_layout()